# 🧪 Model Evaluation & Inference

Test trained models and evaluate performance.

## Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / 'src'))

import torch
import numpy as np
import pandas as pd
from datasets import load_from_disk
from transformers import Wav2Vec2Processor
import matplotlib.pyplot as plt
import seaborn as sns

from src.train import QiraatMTLModel
from src.preprocess import load_audio_safe, normalize_audio_level
from src.tajweed_rules import TajweedDetector

sns.set_style('whitegrid')
print("✅ Setup complete")

## Load Trained Model

In [ ]:
# Check for checkpoints
checkpoint_dir = Path('./checkpoints')
checkpoints = list(checkpoint_dir.glob('*.pt')) if checkpoint_dir.exists() else []

if checkpoints:
    print(f"📁 Found {len(checkpoints)} checkpoint(s):\n")
    for ckpt in sorted(checkpoints):
        size = ckpt.stat().st_size / 1e9
        print(f"  • {ckpt.name} ({size:.2f}GB)")
    
    # Load latest
    latest_ckpt = sorted(checkpoints)[-1]
    print(f"\nLoading: {latest_ckpt.name}")
    
    model = QiraatMTLModel()
    checkpoint = torch.load(latest_ckpt, map_location='cpu')
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    
    print(f"✅ Model loaded on {device}")
else:
    print("⚠️  No checkpoints found")
    print("Please train the model first with: python src/train.py")

## Test Set Evaluation

In [ ]:
# Load test dataset
try:
    dataset_dict = load_from_disk('./data/cache/dataset_splits')
    test_dataset = dataset_dict['test']
    
    print(f"📊 Test Set: {len(test_dataset)} samples")
    
    # Show distribution
    qiraat_counts = {}
    for q in test_dataset['qiraat']:
        qiraat_counts[q] = qiraat_counts.get(q, 0) + 1
    
    print("\n Distribution:")
    for q, count in qiraat_counts.items():
        pct = (count / len(test_dataset)) * 100
        print(f"  {q}: {count} ({pct:.1f}%)")
except Exception as e:
    print(f"⚠️  Error loading test set: {e}")

## Single Sample Inference

In [ ]:
if 'model' in locals():
    # Load sample audio
    sample_path = Path('./data/raw/hafs/001_001_hafs.wav')
    
    if sample_path.exists():
        print(f"📂 Inferring on: {sample_path.name}\n")
        
        audio, sr = load_audio_safe(sample_path, target_sr=16000)
        audio = normalize_audio_level(audio, target_db=-20.0)
        
        # Prepare input
        audio_tensor = torch.from_numpy(audio).float().unsqueeze(0).to(model.device)
        
        # Inference
        with torch.no_grad():
            outputs = model(audio_tensor)
        
        # Task 1: Transcription (logits)
        trans_logits = outputs['transcription']
        print(f"Transcription Output Shape: {trans_logits.shape}")
        print(f"  (batch, time_steps, vocab_size)")
        
        # Task 2: Qira'at Classification
        qiraat_logits = outputs['qiraat']
        qiraat_probs = torch.softmax(qiraat_logits, dim=1)
        qiraat_pred = qiraat_probs.argmax(dim=1).item()
        qiraat_names = ['Hafs', 'Warsh']
        predicted_qiraat = qiraat_names[qiraat_pred]
        confidence = qiraat_probs[0, qiraat_pred].item()
        
        print(f"\nQira'at Classification:")
        print(f"  Predicted: {predicted_qiraat}")
        print(f"  Confidence: {confidence*100:.1f}%")
        print(f"  Probabilities:")
        for i, name in enumerate(qiraat_names):
            prob = qiraat_probs[0, i].item()
            print(f"    {name}: {prob*100:.1f}%")
        
        # Task 3: Tajweed Scoring
        tajweed_logits = outputs['tajweed']
        tajweed_scores = torch.sigmoid(tajweed_logits) * 100
        
        print(f"\nTajweed Scores:")
        rule_names = ['Ghunnah', 'Idgham', 'Imalah', 'Lam Tafkhim', 'Qasr/Madd', 'Hamza']
        for i, name in enumerate(rule_names[:tajweed_scores.shape[1]]):
            score = tajweed_scores[0, i].item()
            print(f"  {name}: {score:.1f}/100")
    else:
        print(f"⚠️  Sample file not found: {sample_path}")

## Batch Inference on Test Set

In [ ]:
if 'model' in locals() and 'test_dataset' in locals():
    print("🚀 Running batch inference on test set...\n")
    
    predictions = []
    qiraat_names = ['Hafs', 'Warsh']
    
    # Process first 10 samples (for speed)
    for i in range(min(10, len(test_dataset))):
        sample = test_dataset[i]
        
        try:
            # Load audio
            audio_path = sample['audio_path']
            audio, sr = load_audio_safe(audio_path, target_sr=16000)
            audio = normalize_audio_level(audio, target_db=-20.0)
            
            # Inference
            audio_tensor = torch.from_numpy(audio).float().unsqueeze(0).to(model.device)
            
            with torch.no_grad():
                outputs = model(audio_tensor)
            
            # Get predictions
            qiraat_probs = torch.softmax(outputs['qiraat'], dim=1)
            pred_qiraat_idx = qiraat_probs.argmax(dim=1).item()
            pred_qiraat = qiraat_names[pred_qiraat_idx]
            confidence = qiraat_probs[0, pred_qiraat_idx].item()
            
            predictions.append({
                'File': Path(audio_path).name,
                'Ground Truth': sample['qiraat'],
                'Predicted': pred_qiraat,
                'Confidence': f"{confidence*100:.1f}%",
                'Correct': pred_qiraat == sample['qiraat'],
            })
        except Exception as e:
            print(f"  ⚠️  Error processing sample {i}: {e}")
    
    # Create DataFrame
    df = pd.DataFrame(predictions)
    accuracy = (df['Correct'].sum() / len(df)) * 100
    
    print(df.to_string(index=False))
    print(f"\n📊 Accuracy: {accuracy:.1f}%")

## Performance Metrics

In [ ]:
if 'df' in locals():
    from sklearn.metrics import confusion_matrix, classification_report
    
    # Confusion matrix
    cm = confusion_matrix(df['Ground Truth'], df['Predicted'], labels=['Hafs', 'Warsh'])
    
    print("📈 Performance Metrics\n")
    print("Confusion Matrix:")
    print(f"           Predicted Hafs  Predicted Warsh")
    print(f"True Hafs       {cm[0,0]}               {cm[0,1]}")
    print(f"True Warsh      {cm[1,0]}               {cm[1,1]}")
    
    # Classification report
    print(f"\n" + classification_report(df['Ground Truth'], df['Predicted']))

## Visualize Predictions

In [ ]:
if 'df' in locals():
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Confusion Matrix Heatmap
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
                xticklabels=['Hafs', 'Warsh'],
                yticklabels=['Hafs', 'Warsh'])
    axes[0].set_title('Confusion Matrix')
    axes[0].set_ylabel('True Label')
    axes[0].set_xlabel('Predicted Label')
    
    # Accuracy by Qiraat
    accuracy_by_qiraat = df.groupby('Ground Truth')['Correct'].apply(
        lambda x: (x.sum() / len(x)) * 100
    )
    
    colors = ['#2ecc71' if acc > 80 else '#f39c12' for acc in accuracy_by_qiraat.values]
    axes[1].bar(accuracy_by_qiraat.index, accuracy_by_qiraat.values, color=colors)
    axes[1].set_ylabel('Accuracy (%)')
    axes[1].set_title('Accuracy by Qira\'at')
    axes[1].set_ylim(0, 100)
    for i, (name, acc) in enumerate(zip(accuracy_by_qiraat.index, accuracy_by_qiraat.values)):
        axes[1].text(i, acc + 2, f'{acc:.1f}%', ha='center')
    
    plt.tight_layout()
    plt.show()

## Expected Results Summary

After training on sufficient data:

| Metric | Expected Value |
|--------|----------------|
| **Qira'at Accuracy** | 92-95% |
| **Hafs Precision** | 91-94% |
| **Warsh Recall** | 90-93% |
| **F1-Score** | 0.91-0.93 |
| **Tajweed Detection** | 85-90% per rule |
| **Ghunnah Duration Error** | ±5ms |

## Troubleshooting

**If accuracy is low:**
1. Check data quality (audio corruption?)
2. Verify metadata is correct
3. Increase training epochs
4. Reduce learning rate
5. Check VRAM usage (gradient accumulation if needed)